# Beam-BC Validation
Tests frozen v39 BC augmented by official information-set search against frozen v39 BC. No training or submission.
Variants: `bc` (no oracle), `beam_generic` (generic heuristic beam), `beam_hybrid` (tactical heuristic + adversarial opponent response, min aggregation). All variants share the identical paired eval schedule.

In [ ]:
SMOKE_MODE = False
N_GAMES = 20 if SMOKE_MODE else 200
N_WORKERS = 4
ORACLE = {'n_determinizations': 1 if SMOKE_MODE else 2, 'top_k': 3 if SMOKE_MODE else 4, 'roll_depth': 6 if SMOKE_MODE else 12, 'target_temperature': 1.0}
ORACLE_HYBRID = dict(ORACLE, heuristic_mode='tactical', enable_opponent_response=True, opponent_top_k=2, opponent_roll_depth=6, opponent_aggregation='min')
ORACLE_STATES_PER_GAME = 1 if SMOKE_MODE else 2
print(f'beam-BC smoke={SMOKE_MODE} games={N_GAMES} workers={N_WORKERS} oracle={ORACLE} hybrid={ORACLE_HYBRID}')

In [ ]:
import glob, json, multiprocessing as mp, os, sys, time
from concurrent.futures import ProcessPoolExecutor
import numpy as np
source_markers = glob.glob('/kaggle/input/**/hardread-tcg-search-src-v*/src/src/__init__.py', recursive=True)
if len(source_markers) != 1: raise RuntimeError(f'Expected one search source mount: {source_markers}')
source_mount = os.path.dirname(os.path.dirname(os.path.dirname(source_markers[0])))
src_root = os.path.join(source_mount, 'src')
cg_root = os.path.join(source_mount, 'cg')
config_root = os.path.join(source_mount, 'configs', 'configs')
for root in (src_root, cg_root): sys.path.insert(0, root)
CONFIG_DIR = config_root
weight_matches = glob.glob('/kaggle/input/**/hardread-tcg-bc-v39/weights.npz', recursive=True)
if len(weight_matches) != 1: raise RuntimeError(f'Expected exactly one v39 weights.npz: {weight_matches}')
from src.policy_numpy import load_params
from src.selfplay import load_decks, make_eval_schedule, tally_eval_results
from src.rollout_worker import init_worker, play_game
import src.search_oracle as _search_oracle, src.rollout_worker as _rollout_worker
print('search source:', _search_oracle.__file__, _rollout_worker.__file__)
assert 'hardread-tcg-search-src-v' in _search_oracle.__file__ and 'hardread-tcg-search-src-v' in _rollout_worker.__file__
params = load_params(weight_matches[0])
decks = load_decks(CONFIG_DIR, ['Lucario', 'Crustle', 'Alakazam'])
base_tasks = make_eval_schedule(list(decks), decks, N_GAMES, 880001)
def run_variant(name, oracle_cfg):
    tasks = [dict(t, record_transitions=False, oracle_states_per_game=(ORACLE_STATES_PER_GAME if oracle_cfg else 0), execute_oracle=bool(oracle_cfg)) for t in base_tasks]
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp.get_context('spawn'), initializer=init_worker, initargs=(params, [params], 64, 512, 0.0, 0.99, oracle_cfg)) as executor:
        results = list(executor.map(play_game, tasks))
    elapsed = time.time() - t0
    score = tally_eval_results(results)
    fail_reasons = {}
    for r in results:
        for reason, count in r.get('oracle_fail_reasons', {}).items(): fail_reasons[reason] = fail_reasons.get(reason, 0) + count
    oracle = {'attempts': sum(r.get('oracle_attempts', 0) for r in results), 'successes': sum(r.get('oracle_successes', 0) for r in results), 'errors': sum(r.get('oracle_errors', 0) for r in results), 'examples': sum(len(r.get('oracle_examples', [])) for r in results), 'fail_reasons': fail_reasons}
    oracle['success_rate'] = oracle['successes'] / max(1, oracle['attempts'])
    return {'score': score, 'oracle': oracle, 'elapsed_s': elapsed, 'games_per_s': N_GAMES / max(elapsed, 1e-9), 'name': name}
metrics = {'bc': run_variant('bc', None), 'beam_generic': run_variant('beam_generic', ORACLE), 'beam_hybrid': run_variant('beam_hybrid', ORACLE_HYBRID), 'config': {'smoke': SMOKE_MODE, 'games': N_GAMES, 'workers': N_WORKERS, 'oracle': ORACLE, 'oracle_hybrid': ORACLE_HYBRID}}
os.makedirs('/kaggle/working/beam_bc_eval', exist_ok=True)
with open('/kaggle/working/beam_bc_eval/metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))